# 模运算 完整教程：从取余数到信息编码

## 📚 教学目标
1. 理解 **模运算 (Modular Arithmetic)** 的基本定义与核心性质
2. 掌握 **同余关系 (Congruence)** 在加法、乘法中的保持性
3. 学会利用 **模不变量 (Mod Invariant)** 证明"不可能"命题
4. 运用模运算解决经典脑筋急转弯：囚犯猜帽子、变色龙、约瑟夫环
5. 用 Python 模拟验证每一个分析结论

**参考书**：《A Practical Guide to Quantitative Finance Interviews》(Xinfeng Zhou) 第2章
**教学策略**：先用极小数据集让你"看见"每一步计算，再用 Monte Carlo 模拟验证分析解

---

## 1. 场景设定：为什么模运算在面试中如此重要？

### 🎯 问题

在量化面试中，你会遇到大量看似复杂的脑筋急转弯。很多问题的关键突破口在于发现一个**不变量 (invariant)**——而模运算就是构造不变量最强大的工具之一。

| 问题类型 | 模运算的作用 |
|---------|-------------|
| 囚犯猜帽子 | 利用 $\text{mod } k$ 编码全局信息 |
| 变色龙问题 | 构造 $\text{mod } 3$ 不变量证明不可能 |
| 约瑟夫环 | 递推公式中的 $\text{mod } n$ |
| 最后一位数字 | 幂次的周期性 $\text{mod } 10$ |

### 💡 核心思路

模运算的本质是**只关注余数，忽略商**。这种"降维"能力让我们从复杂问题中提取出简单的结构。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import product

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 模运算基础

### 📐 定义

给定正整数 $n$，整数 $a$ 除以 $n$ 的**余数**记为：

$$a \bmod n = r \quad \text{其中} \quad a = qn + r, \; 0 \le r < n$$

如果 $a \bmod n = b \bmod n$，我们说 $a$ 与 $b$ **模 $n$ 同余**，记为：

$$a \equiv b \pmod{n}$$

### 📐 核心性质

模运算对加法和乘法**封闭**：

$$\text{加法: } (a + b) \bmod n = [(a \bmod n) + (b \bmod n)] \bmod n$$

$$\text{乘法: } (a \times b) \bmod n = [(a \bmod n) \times (b \bmod n)] \bmod n$$

$$\text{幂次: } a^k \bmod n = (a \bmod n)^k \bmod n$$

### 💡 直觉

想象一个**时钟**：12 点和 0 点是同一个位置。时钟上的运算就是 $\text{mod } 12$。
- $15 \text{ 点} = 3 \text{ 点}$，因为 $15 \equiv 3 \pmod{12}$
- $8 \text{ 点} + 7 \text{ 小时} = 3 \text{ 点}$，因为 $(8 + 7) \bmod 12 = 3$

In [ ]:
# ========== 步骤 1: 模运算基本性质验证 ==========
print("📊 步骤 1: 模运算基本性质")
print("=" * 55)

n = 7  # 模数
a, b = 23, 15

print(f"  模数 n = {n}")
print(f"  a = {a}, b = {b}")
print(f"  a mod {n} = {a % n}, b mod {n} = {b % n}")

# 加法性质
print(f"\n📊 步骤 1a: 加法性质")
lhs_add = (a + b) % n
rhs_add = ((a % n) + (b % n)) % n
print(f"  (a + b) mod n = ({a} + {b}) mod {n} = {a + b} mod {n} = {lhs_add}")
print(f"  [(a mod n) + (b mod n)] mod n = [{a % n} + {b % n}] mod {n} = {(a % n) + (b % n)} mod {n} = {rhs_add}")
print(f"  结果一致: {lhs_add == rhs_add} ✅")

# 乘法性质
print(f"\n📊 步骤 1b: 乘法性质")
lhs_mul = (a * b) % n
rhs_mul = ((a % n) * (b % n)) % n
print(f"  (a × b) mod n = ({a} × {b}) mod {n} = {a * b} mod {n} = {lhs_mul}")
print(f"  [(a mod n) × (b mod n)] mod n = [{a % n} × {b % n}] mod {n} = {(a % n) * (b % n)} mod {n} = {rhs_mul}")
print(f"  结果一致: {lhs_mul == rhs_mul} ✅")

# 幂次性质
print(f"\n📊 步骤 1c: 幂次性质")
k = 5
lhs_pow = (a ** k) % n
rhs_pow = pow(a % n, k, n)  # Python 内置模幂运算
print(f"  a^k mod n = {a}^{k} mod {n} = {a**k} mod {n} = {lhs_pow}")
print(f"  (a mod n)^k mod n = {a % n}^{k} mod {n} = {(a % n)**k} mod {n} = {rhs_pow}")
print(f"  结果一致: {lhs_pow == rhs_pow} ✅")

print(f"\n💡 这些性质让我们可以'先取余再运算'，避免处理巨大的数字")

In [ ]:
# ========== 步骤 2: 同余类可视化 (模运算时钟) ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- 左图: mod 12 时钟 ---
ax1 = axes[0]
n_clock = 12
theta = np.linspace(np.pi/2, np.pi/2 - 2*np.pi, n_clock, endpoint=False)
x_pos = np.cos(theta)
y_pos = np.sin(theta)

# 画圆
circle = plt.Circle((0, 0), 1.0, fill=False, color='gray', linewidth=2)
ax1.add_patch(circle)

# 标注数字
for i in range(n_clock):
    ax1.plot(x_pos[i], y_pos[i], 'o', color='steelblue', markersize=20, zorder=5)
    ax1.text(x_pos[i], y_pos[i], str(i), ha='center', va='center',
             fontsize=11, fontweight='bold', color='white', zorder=6)

# 标注 15 mod 12 = 3
ax1.annotate('15 mod 12 = 3', xy=(x_pos[3], y_pos[3]),
             xytext=(1.5, 0.8), fontsize=12, color='#e74c3c',
             arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))

# 标注 27 mod 12 = 3
ax1.annotate('27 mod 12 = 3', xy=(x_pos[3], y_pos[3]),
             xytext=(1.5, 0.3), fontsize=12, color='#e67e22',
             arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2))

ax1.set_xlim(-1.8, 2.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_aspect('equal')
ax1.set_title('Modular Clock (mod 12)', fontsize=14, fontweight='bold')
ax1.axis('off')

# --- 右图: mod 7 的同余类 ---
ax2 = axes[1]
n_mod = 7
numbers = list(range(0, 28))
classes = {i: [] for i in range(n_mod)}
for num in numbers:
    classes[num % n_mod].append(num)

colors_list = plt.cm.Set2(np.linspace(0, 1, n_mod))
for cls_id in range(n_mod):
    y_val = n_mod - cls_id - 1
    members = classes[cls_id]
    for j, m in enumerate(members):
        ax2.plot(j, y_val, 's', color=colors_list[cls_id], markersize=22)
        ax2.text(j, y_val, str(m), ha='center', va='center',
                 fontsize=9, fontweight='bold')
    ax2.text(-0.8, y_val, f'[{cls_id}]', ha='center', va='center',
             fontsize=12, fontweight='bold', color=colors_list[cls_id])

ax2.set_xlim(-1.5, len(classes[0]) + 0.5)
ax2.set_ylim(-0.8, n_mod + 0.3)
ax2.set_xlabel('Members of Congruence Class', fontsize=12)
ax2.set_ylabel('Class [r]', fontsize=12)
ax2.set_title(f'Congruence Classes mod {n_mod}', fontsize=14, fontweight='bold')
ax2.set_yticks([])
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：模 12 时钟 —— 15 和 27 都指向 3 的位置，因为它们 mod 12 都等于 3")
print(f"  右图：mod 7 的 7 个同余类 —— 同一行的数字除以 7 的余数相同")
print(f"       例如 [2] 类: 2, 9, 16, 23 都满足 n mod 7 = 2")

---

## 3. 经典问题一：100 囚犯猜帽子

### 🎯 问题

100 个囚犯排成一列。每人头上随机放一顶帽子，颜色为 $0, 1, \ldots, k-1$ 中的一种（$k$ 种颜色）。每个人能看到**前面所有人**的帽子，但看不到自己的和后面人的。

从最后一个人开始，每人必须大声猜自己帽子的颜色。所有人都能听到之前猜过的答案。

**问**：能否设计一个策略，保证**至少 99 人**猜对？

### 📐 策略（模运算）

设第 $i$ 个人的帽子颜色为 $h_i$，$i = 1, 2, \ldots, 100$。

$$S = \sum_{i=1}^{100} h_i \pmod{k}$$

**策略**：
1. **第 100 人**（最后一个，能看到前 99 人的帽子）：
   - 计算 $S_{99} = \sum_{i=1}^{99} h_i \bmod k$
   - 大声报出 $S_{99}$（而不是猜自己的帽子）
   - 这等于宣布了 $S_{99} \bmod k$ 的值

2. **第 99 人**：
   - 已知 $S_{99}$（第 100 人报出的值）
   - 能看到前面 98 人：计算 $S_{98} = \sum_{i=1}^{98} h_i \bmod k$
   - 自己的帽子：$h_{99} = (S_{99} - S_{98}) \bmod k$ ✅

3. **第 $j$ 人**（一般情况）：
   - 已知：$S_{99}$，以及第 $j+1$ 到第 99 人的真实帽子（因为他们都猜对了）
   - 能看到前面 $j-1$ 人的帽子
   - 自己的帽子：$h_j = (S_{99} - \sum_{i \ne j} h_i) \bmod k$ ✅

### 💡 关键洞察

第 100 人牺牲自己（他的猜测概率只有 $1/k$），用来传递**整个帽子序列的模 $k$ 信息**。其余 99 人利用这个信息 + 可见信息，**确定性地**推断出自己的帽子颜色。

In [ ]:
# ========== 步骤 3: 微型例子 —— 5 个囚犯，3 种颜色 ==========
print("📊 步骤 3: 微型例子 —— 5 个囚犯，3 种颜色")
print("=" * 55)

n_prisoners = 5
k_colors = 3
np.random.seed(42)
hats = np.random.randint(0, k_colors, n_prisoners)

print(f"  囚犯编号:  {list(range(1, n_prisoners + 1))}")
print(f"  帽子颜色:  {list(hats)}")
print(f"  颜色含义:  0=红, 1=绿, 2=蓝")

# 第5人(最后)能看到前4人
S_front = sum(hats[:n_prisoners - 1]) % k_colors
print(f"\n  --- 第 {n_prisoners} 人（最后一个）开始 ---")
print(f"  看到前 {n_prisoners-1} 人: {list(hats[:n_prisoners-1])}")
print(f"  计算 S = ({' + '.join(map(str, hats[:n_prisoners-1]))}) mod {k_colors} = {sum(hats[:n_prisoners-1])} mod {k_colors} = {S_front}")
print(f"  报出: {S_front}")
print(f"  实际帽子: {hats[n_prisoners-1]}, 猜对? {S_front == hats[n_prisoners-1]}")

# 其余人逆序推断
guesses = [0] * n_prisoners
guesses[n_prisoners - 1] = S_front  # 第5人报的数
known_sum = S_front  # 这是前4人帽子的 mod k 总和

print(f"\n  --- 第 4 到第 1 人依次推断 ---")
for j in range(n_prisoners - 2, -1, -1):  # j = 3, 2, 1, 0
    # 已知信息: known_sum = sum(hats[0..n-2]) mod k
    # 能看到前面的人: hats[0..j-1]
    # 已知后面的人(已猜对): hats[j+1..n-2]
    visible = list(hats[:j])  # 前面的人
    heard_correct = list(hats[j+1:n_prisoners-1])  # 后面已猜对的人
    others_sum = (sum(visible) + sum(heard_correct)) % k_colors
    my_hat = (known_sum - others_sum) % k_colors
    guesses[j] = my_hat
    
    print(f"\n  第 {j+1} 人:")
    print(f"    看到前面: {visible}")
    print(f"    听到后面(已猜对): {heard_correct}")
    print(f"    已知总和 S = {known_sum}")
    print(f"    其他人总和 mod {k_colors} = ({'+'.join(map(str, visible + heard_correct)) if visible + heard_correct else '0'}) mod {k_colors} = {others_sum}")
    print(f"    我的帽子 = (S - 其他人) mod {k_colors} = ({known_sum} - {others_sum}) mod {k_colors} = {my_hat}")
    print(f"    实际帽子: {hats[j]}, 猜对? {my_hat == hats[j]} {'✅' if my_hat == hats[j] else '❌'}")

correct = sum(g == h for g, h in zip(guesses, hats))
print(f"\n📊 结果: {correct}/{n_prisoners} 人猜对")
print(f"💡 只有最后一个人可能猜错（他牺牲自己来传递模信息），其余人都确定性猜对")

In [ ]:
# ========== 步骤 4: Monte Carlo 模拟验证（100 囚犯，k 种颜色）==========
print("📊 步骤 4: Monte Carlo 模拟验证")
print("=" * 55)

def simulate_prisoner_hats(n_prisoners, k_colors, n_trials=10000):
    """模拟囚犯猜帽子策略"""
    correct_counts = []  # 每次试验猜对的人数
    last_person_correct = 0  # 最后一个人猜对的次数
    
    for _ in range(n_trials):
        hats = np.random.randint(0, k_colors, n_prisoners)
        
        # 最后一个人报出前 n-1 人的帽子总和 mod k
        S = sum(hats[:n_prisoners - 1]) % k_colors
        
        # 最后一个人是否碰巧猜对自己
        if S == hats[n_prisoners - 1]:
            last_person_correct += 1
        
        # 其余人确定性推断 —— 一定猜对
        # 所以猜对人数 = 99 + (最后一个人碰巧对了就是1, 否则0)
        correct = (n_prisoners - 1) + (1 if S == hats[n_prisoners - 1] else 0)
        correct_counts.append(correct)
    
    return np.array(correct_counts), last_person_correct

n_trials = 10000
results = {}
for k in [2, 3, 5, 10]:
    counts, last_correct = simulate_prisoner_hats(100, k, n_trials)
    results[k] = {
        'min_correct': counts.min(),
        'max_correct': counts.max(),
        'mean_correct': counts.mean(),
        'always_99_plus': (counts >= 99).mean(),
        'last_person_prob': last_correct / n_trials
    }

print(f"  100 囚犯，{n_trials} 次模拟：")
print(f"  {'颜色数k':>8} {'最少猜对':>10} {'最多猜对':>10} {'平均猜对':>10} {'>=99人概率':>12} {'最后人猜对率':>14}")
print(f"  {'─'*68}")
for k, r in results.items():
    print(f"  {k:>8d} {r['min_correct']:>10d} {r['max_correct']:>10d} {r['mean_correct']:>10.2f} {r['always_99_plus']:>12.4f} {r['last_person_prob']:>14.4f}")

print(f"\n💡 关键观察：")
print(f"  1. 无论 k 多大，至少 99 人猜对的概率 = 100%")
print(f"  2. 最后一个人猜对的概率 ≈ 1/k（纯靠运气）")
print(f"  3. 平均猜对人数 ≈ 99 + 1/k")

In [ ]:
# ========== 步骤 5: 可视化囚犯策略 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 不同 k 下的猜对人数分布 ---
ax1 = axes[0]
k_values = [2, 3, 5, 10]
bar_width = 0.2
colors_k = ['steelblue', '#2ecc71', '#e67e22', '#e74c3c']

for idx, k in enumerate(k_values):
    counts, _ = simulate_prisoner_hats(100, k, 5000)
    unique, freq = np.unique(counts, return_counts=True)
    ax1.bar(unique + idx * bar_width - 0.3, freq / len(counts),
            width=bar_width, color=colors_k[idx], alpha=0.8,
            label=f'k={k}', edgecolor='black')

ax1.set_xlabel('Number of Correct Guesses', fontsize=12)
ax1.set_ylabel('Probability', fontsize=12)
ax1.set_title('Distribution of Correct Guesses (100 Prisoners)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 右图: 最后一个人猜对概率 vs 1/k ---
ax2 = axes[1]
k_range = list(range(2, 21))
empirical_probs = []
for k in k_range:
    _, last_correct = simulate_prisoner_hats(100, k, 5000)
    empirical_probs.append(last_correct / 5000)

ax2.plot(k_range, empirical_probs, 'o-', color='steelblue', markersize=6,
         linewidth=2, label='Simulation')
ax2.plot(k_range, [1/k for k in k_range], 's--', color='#e74c3c',
         markersize=5, linewidth=2, label='Theoretical (1/k)')

ax2.set_xlabel('Number of Colors (k)', fontsize=12)
ax2.set_ylabel('P(Last Prisoner Correct)', fontsize=12)
ax2.set_title('Last Prisoner: Empirical vs Theoretical', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：猜对人数只可能是 99 或 100，因为策略保证至少 99 人正确")
print(f"  右图：最后一个人猜对概率精确等于 1/k —— 他的猜测本质是随机的")

---

## 4. 经典问题二：变色龙问题

### 🎯 问题

一个岛上有 **13 只红色**、**15 只绿色**、**17 只蓝色** 变色龙。

**规则**：当两只**不同颜色**的变色龙相遇时，它们都变成**第三种颜色**。
- 例如：1 红 + 1 绿 → 2 蓝

**问**：是否有可能最终所有变色龙变成同一种颜色？

### 📐 分析：模 3 不变量

设三种颜色的数量分别为 $(R, G, B)$。初始状态 $(13, 15, 17)$。

当红绿相遇时：$(R, G, B) \to (R-1, G-1, B+2)$

观察差值的变化：

$$R - G: \quad (R-1) - (G-1) = R - G \quad (\text{不变})$$

等等，差值不完全不变。让我们看 **mod 3**：

任意一次相遇：某两种颜色各 $-1$，第三种 $+2$。所以：

$$\Delta R \equiv 0 \text{ or } -1 \text{ or } +2 \equiv -1 \pmod{3}$$

关键不变量：**每种颜色的数量 mod 3 的变化相同**。

具体地，在任何一次相遇中，$(R \bmod 3, G \bmod 3, B \bmod 3)$ 的变化是：
- 两个分量 $-1 \pmod{3}$，一个分量 $+2 \equiv -1 \pmod{3}$

所以所有三个分量都变化 $-1 \pmod{3}$，意味着：

$$R - G \equiv \text{常数} \pmod{3}, \quad G - B \equiv \text{常数} \pmod{3}$$

初始状态：
- $R - G = 13 - 15 = -2 \equiv 1 \pmod{3}$
- $G - B = 15 - 17 = -2 \equiv 1 \pmod{3}$

如果所有变色龙变成同一种颜色（比如全红），则 $(45, 0, 0)$：
- $R - G = 45 - 0 = 45 \equiv 0 \pmod{3}$

但初始状态 $R - G \equiv 1 \pmod{3}$，**不等于 0**！

### 🎯 结论

$$\text{不可能！} \quad \text{因为 mod 3 不变量被违反}$$

In [ ]:
# ========== 步骤 6: 验证变色龙问题的 mod 3 不变量 ==========
print("📊 步骤 6: 验证变色龙 mod 3 不变量")
print("=" * 55)

R0, G0, B0 = 13, 15, 17
total = R0 + G0 + B0

print(f"  初始状态: (R, G, B) = ({R0}, {G0}, {B0})")
print(f"  总数: {total}")
print(f"  R mod 3 = {R0 % 3}")
print(f"  G mod 3 = {G0 % 3}")
print(f"  B mod 3 = {B0 % 3}")
print(f"  (R - G) mod 3 = ({R0} - {G0}) mod 3 = {(R0 - G0) % 3}")
print(f"  (G - B) mod 3 = ({G0} - {B0}) mod 3 = {(G0 - B0) % 3}")

# 验证所有三种相遇类型都保持 (R-G) mod 3 不变
print(f"\n📊 验证三种相遇都保持不变量:")
meetings = [
    ("红+绿→蓝蓝", (-1, -1, 2)),
    ("红+蓝→绿绿", (-1, 2, -1)),
    ("绿+蓝→红红", (2, -1, -1))
]

for name, (dr, dg, db) in meetings:
    R1, G1, B1 = R0 + dr, G0 + dg, B0 + db
    diff_rg_before = (R0 - G0) % 3
    diff_rg_after = (R1 - G1) % 3
    diff_gb_before = (G0 - B0) % 3
    diff_gb_after = (G1 - B1) % 3
    print(f"  {name}: ({R0},{G0},{B0}) → ({R1},{G1},{B1})")
    print(f"    (R-G) mod 3: {diff_rg_before} → {diff_rg_after}  不变? {diff_rg_before == diff_rg_after} ✅")
    print(f"    (G-B) mod 3: {diff_gb_before} → {diff_gb_after}  不变? {diff_gb_before == diff_gb_after} ✅")

# 检验终态
print(f"\n📊 检验三种可能的终态:")
for name, (Rf, Gf, Bf) in [("全红", (45,0,0)), ("全绿", (0,45,0)), ("全蓝", (0,0,45))]:
    rg = (Rf - Gf) % 3
    gb = (Gf - Bf) % 3
    print(f"  {name} ({Rf},{Gf},{Bf}): (R-G) mod 3 = {rg}, (G-B) mod 3 = {gb}")
    match = (rg == (R0-G0)%3) and (gb == (G0-B0)%3)
    print(f"    与初始不变量匹配? {match} {'✅ 可能' if match else '❌ 不可能'}")

print(f"\n💡 三种终态的不变量都与初始状态不匹配 → 不可能所有变色龙变成同一种颜色！")

In [ ]:
# ========== 步骤 7: 模拟变色龙随机相遇过程 ==========
print("📊 步骤 7: 模拟变色龙随机相遇 (追踪不变量)")
print("=" * 55)

def simulate_chameleons(R0, G0, B0, n_steps=500):
    """模拟变色龙随机相遇过程"""
    R, G, B = R0, G0, B0
    history = [(R, G, B)]
    
    for step in range(n_steps):
        # 随机选择两只不同颜色的变色龙
        possible_meetings = []
        if R > 0 and G > 0:
            possible_meetings.append('RG')
        if R > 0 and B > 0:
            possible_meetings.append('RB')
        if G > 0 and B > 0:
            possible_meetings.append('GB')
        
        if not possible_meetings:
            break  # 已经全部同色
        
        meeting = np.random.choice(possible_meetings)
        if meeting == 'RG':
            R -= 1; G -= 1; B += 2
        elif meeting == 'RB':
            R -= 1; B -= 1; G += 2
        else:  # GB
            G -= 1; B -= 1; R += 2
        
        history.append((R, G, B))
    
    return history

np.random.seed(42)
history = simulate_chameleons(13, 15, 17, n_steps=1000)

# 验证不变量在整个过程中保持
rg_mod3 = [(r - g) % 3 for r, g, b in history]
gb_mod3 = [(g - b) % 3 for r, g, b in history]

print(f"  模拟步数: {len(history) - 1}")
print(f"  最终状态: (R, G, B) = {history[-1]}")
print(f"  (R-G) mod 3 在所有步骤中的值: {set(rg_mod3)}")
print(f"  (G-B) mod 3 在所有步骤中的值: {set(gb_mod3)}")
print(f"  不变量始终保持? {len(set(rg_mod3)) == 1 and len(set(gb_mod3)) == 1} ✅")

# 检查最小的非零颜色数
min_nonzero = []
for r, g, b in history:
    vals = [x for x in [r, g, b] if x > 0]
    if vals:
        min_nonzero.append(min(vals))

print(f"\n  过程中某种颜色的最小非零数量: {min(min_nonzero)}")
print(f"  最终是否有某种颜色为0? {0 in history[-1]}")
print(f"\n💡 即使经过大量随机相遇，不变量始终保持")
print(f"   无论怎么模拟，都不可能达到 (45,0,0), (0,45,0) 或 (0,0,45)")

In [ ]:
# ========== 步骤 8: 变色龙过程可视化 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 三种颜色数量随时间变化 ---
ax1 = axes[0]
Rs = [h[0] for h in history]
Gs = [h[1] for h in history]
Bs = [h[2] for h in history]
steps = range(len(history))

ax1.plot(steps, Rs, '-', color='#e74c3c', linewidth=1.5, alpha=0.8, label='Red')
ax1.plot(steps, Gs, '-', color='#2ecc71', linewidth=1.5, alpha=0.8, label='Green')
ax1.plot(steps, Bs, '-', color='steelblue', linewidth=1.5, alpha=0.8, label='Blue')
ax1.axhline(y=45, color='gray', linestyle='--', alpha=0.5, label='Total = 45')
ax1.axhline(y=0, color='black', linewidth=0.5)

ax1.set_xlabel('Meeting Step', fontsize=12)
ax1.set_ylabel('Number of Chameleons', fontsize=12)
ax1.set_title('Chameleon Population Over Time', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# --- 右图: (R-G) mod 3 不变量 ---
ax2 = axes[1]
ax2.plot(steps, rg_mod3, 'o-', color='#e67e22', markersize=2, linewidth=1,
         alpha=0.8, label='(R - G) mod 3')
ax2.plot(steps, gb_mod3, 's-', color='#9b59b6', markersize=2, linewidth=1,
         alpha=0.8, label='(G - B) mod 3')

ax2.set_xlabel('Meeting Step', fontsize=12)
ax2.set_ylabel('Invariant Value', fontsize=12)
ax2.set_title('Mod 3 Invariants Remain Constant', fontsize=14, fontweight='bold')
ax2.set_ylim(-0.5, 3.5)
ax2.set_yticks([0, 1, 2])
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

# 注释框
textstr = f'Invariant (R-G) mod 3 = {rg_mod3[0]}\nInvariant (G-B) mod 3 = {gb_mod3[0]}\nTarget needs both = 0'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax2.text(0.98, 0.98, textstr, transform=ax2.transAxes, fontsize=10,
         verticalalignment='top', horizontalalignment='right', bbox=props)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：三种颜色的变色龙数量在随机相遇中不断波动，但总数始终 = 45")
print(f"  右图：(R-G) mod 3 和 (G-B) mod 3 始终保持不变")
print(f"       如果要全部同色，需要不变量 = 0，但实际值是 1 → 永远不可能达到")

---

## 5. 经典问题三：约瑟夫问题

### 🎯 问题

$n$ 个人围成一圈，编号 $0, 1, \ldots, n-1$。从第 0 人开始计数，每数到第 $k$ 个人就将其淘汰。最后剩下的人的编号是什么？

### 📐 递推公式

设 $J(n, k)$ 为 $n$ 人、每 $k$ 个淘汰的幸存者编号，则：

$$J(1, k) = 0$$

$$J(n, k) = [J(n-1, k) + k] \bmod n \quad (n > 1)$$

### 💡 直觉

淘汰第一个人后，剩下 $n-1$ 人形成一个新的环。但编号发生了**偏移**：新的起点从被淘汰人的下一个开始。所以 $n-1$ 人问题的答案需要**加上偏移量 $k$ 再取模 $n$**，就得到 $n$ 人问题的答案。

In [ ]:
# ========== 步骤 9: 约瑟夫问题 —— 微型例子 ==========
print("📊 步骤 9: 约瑟夫问题 微型例子")
print("=" * 55)

n, k = 7, 3
print(f"  n = {n} 人, 每 k = {k} 个淘汰")
print(f"  编号: 0, 1, 2, ..., {n-1}")

# 手动模拟淘汰过程
circle = list(range(n))
idx = 0
elimination_order = []

print(f"\n  淘汰过程:")
step = 1
while len(circle) > 1:
    idx = (idx + k - 1) % len(circle)
    eliminated = circle.pop(idx)
    elimination_order.append(eliminated)
    print(f"    第 {step} 轮: 淘汰 {eliminated}, 剩余 {circle}")
    step += 1
    if idx >= len(circle):
        idx = 0

survivor_sim = circle[0]
print(f"\n  最终幸存者 (模拟): {survivor_sim}")

# 递推公式计算
print(f"\n📊 递推公式计算:")
J = 0  # J(1, k) = 0
print(f"  J(1, {k}) = 0")
for i in range(2, n + 1):
    J = (J + k) % i
    print(f"  J({i}, {k}) = (J({i-1},{k}) + {k}) mod {i} = ({J - k if J >= k else J - k + i} + {k}) mod {i} = {J}")

print(f"\n  递推公式结果: J({n}, {k}) = {J}")
print(f"  模拟结果:     {survivor_sim}")
print(f"  一致? {J == survivor_sim} ✅")

In [ ]:
# ========== 步骤 10: 约瑟夫问题 —— 递推 vs 模拟批量验证 ==========
print("📊 步骤 10: 批量验证递推公式")
print("=" * 55)

def josephus_recursive(n, k):
    """递推公式计算约瑟夫问题"""
    J = 0
    for i in range(2, n + 1):
        J = (J + k) % i
    return J

def josephus_simulate(n, k):
    """直接模拟约瑟夫问题"""
    circle = list(range(n))
    idx = 0
    while len(circle) > 1:
        idx = (idx + k - 1) % len(circle)
        circle.pop(idx)
        if idx >= len(circle):
            idx = 0
    return circle[0]

# 验证 n=2 到 20, k=2 到 5
print(f"  {'n':>4} {'k=2':>6} {'k=3':>6} {'k=4':>6} {'k=5':>6}")
print(f"  {'─'*28}")
all_match = True
for n_test in range(2, 21):
    row = f"  {n_test:>4}"
    for k_test in [2, 3, 4, 5]:
        j_rec = josephus_recursive(n_test, k_test)
        j_sim = josephus_simulate(n_test, k_test)
        if j_rec != j_sim:
            all_match = False
        row += f" {j_rec:>6}"
    print(row)

print(f"\n  所有结果一致? {all_match} ✅")
print(f"\n💡 递推公式 J(n,k) = [J(n-1,k) + k] mod n 完美匹配模拟结果")
print(f"   递推的时间复杂度是 O(n)，远比直接模拟 O(nk) 高效")

In [ ]:
# ========== 步骤 11: 约瑟夫问题可视化 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- 左图: 淘汰顺序的环形可视化 (n=10, k=3) ---
ax1 = axes[0]
n_vis = 10
k_vis = 3
theta_j = np.linspace(np.pi/2, np.pi/2 - 2*np.pi, n_vis, endpoint=False)
x_j = np.cos(theta_j) * 1.0
y_j = np.sin(theta_j) * 1.0

# 模拟获得淘汰顺序
circle = list(range(n_vis))
idx = 0
elim_order = []
while len(circle) > 1:
    idx = (idx + k_vis - 1) % len(circle)
    elim_order.append(circle.pop(idx))
    if idx >= len(circle):
        idx = 0
survivor = circle[0]

# 颜色: 越早淘汰越红，幸存者绿色
cmap_j = plt.cm.Reds(np.linspace(0.3, 0.9, n_vis - 1))
person_colors = {}
for order, person in enumerate(elim_order):
    person_colors[person] = cmap_j[order]
person_colors[survivor] = '#2ecc71'

for i in range(n_vis):
    ax1.plot(x_j[i], y_j[i], 'o', color=person_colors[i], markersize=28,
             markeredgecolor='black', markeredgewidth=1.5, zorder=5)
    ax1.text(x_j[i], y_j[i], str(i), ha='center', va='center',
             fontsize=11, fontweight='bold', zorder=6)
    # 淘汰序号
    if i in elim_order:
        order = elim_order.index(i) + 1
        ax1.text(x_j[i] * 1.3, y_j[i] * 1.3, f'#{order}',
                 ha='center', va='center', fontsize=9, color='gray')

ax1.set_xlim(-1.8, 1.8)
ax1.set_ylim(-1.6, 1.6)
ax1.set_aspect('equal')
ax1.set_title(f'Josephus Problem (n={n_vis}, k={k_vis})\nSurvivor: {survivor}',
              fontsize=14, fontweight='bold')
ax1.axis('off')

# --- 右图: J(n, 2) 的模式 ---
ax2 = axes[1]
n_range = list(range(1, 65))
j_vals = [josephus_recursive(n_val, 2) for n_val in n_range]

# 用不同颜色标注 2 的幂次段
colors_seg = plt.cm.Set2(np.linspace(0, 1, 7))
power = 0
seg_start = 1
for i, n_val in enumerate(n_range):
    if n_val >= 2 ** (power + 1):
        power += 1
    ax2.plot(n_val, j_vals[i], 'o', color=colors_seg[power % len(colors_seg)],
             markersize=5, zorder=5)

ax2.plot(n_range, j_vals, '-', color='gray', linewidth=0.5, alpha=0.5)

# 标注 2 的幂次
for p in range(7):
    n_p = 2 ** p
    if n_p <= 64:
        ax2.axvline(x=n_p, color='gray', linestyle=':', alpha=0.4)
        ax2.text(n_p, max(j_vals) * 0.95, f'$2^{p}$', ha='center',
                 fontsize=9, color='gray')

ax2.set_xlabel('n (Number of People)', fontsize=12)
ax2.set_ylabel('J(n, 2) (Survivor Position)', fontsize=12)
ax2.set_title('Josephus Problem J(n, k=2) Pattern', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：10人/每3个淘汰的环形图。颜色越红 = 越早被淘汰，绿色 = 幸存者")
print(f"  右图：J(n,2) 呈锯齿形，每到 2 的幂次处归零，然后线性增长")
print(f"       这对应封闭公式 J(n,2) = 2L + 1，其中 n = 2^m + L")

---

## 6. 经典问题四：最后一位数字

### 🎯 问题

$7^{2023}$ 的最后一位数字是什么？

### 📐 分析

最后一位数字就是 $7^{2023} \bmod 10$。

利用模运算的幂次性质，我们只需追踪 $7^n \bmod 10$ 的周期：

| $n$ | $7^n$ | $7^n \bmod 10$ |
|-----|-------|----------------|
| 1 | 7 | 7 |
| 2 | 49 | 9 |
| 3 | 343 | 3 |
| 4 | 2401 | 1 |
| 5 | 16807 | 7 |

**周期为 4**：$7, 9, 3, 1, 7, 9, 3, 1, \ldots$

$$7^{2023} \bmod 10 = 7^{2023 \bmod 4} \bmod 10 = 7^3 \bmod 10 = 3$$

### 💡 一般方法

对于 $a^n \bmod m$：
1. 计算 $a, a^2, a^3, \ldots$ 对 $m$ 取模的序列
2. 找到周期 $T$（当结果回到 $a \bmod m$ 时）
3. $a^n \bmod m = a^{n \bmod T} \bmod m$（当 $n \bmod T \ne 0$ 时）
   - 如果 $n \bmod T = 0$，则 $a^n \bmod m = a^T \bmod m$

In [ ]:
# ========== 步骤 12: 幂次的模运算周期 ==========
print("📊 步骤 12: 幂次的模运算周期")
print("=" * 55)

# 找 a^n mod m 的周期
def find_power_cycle(a, m, max_n=100):
    """找到 a^n mod m 的周期"""
    cycle = []
    for n in range(1, max_n + 1):
        val = pow(a, n, m)
        cycle.append(val)
        # 检查是否形成周期
        if n >= 2 and val == cycle[0]:
            period = n - 1
            # 验证周期
            if all(cycle[i] == cycle[i % period] for i in range(period, n)):
                return period, cycle[:period]
    return len(cycle), cycle

# 各底数 mod 10 的周期
print(f"  底数 a 的 a^n mod 10 周期：")
print(f"  {'底数':>4} {'周期':>6} {'循环序列':>30}")
print(f"  {'─'*42}")
for a in range(2, 11):
    period, cycle = find_power_cycle(a, 10)
    print(f"  {a:>4} {period:>6} {str(cycle):>30}")

# 解决 7^2023 mod 10
print(f"\n📊 解决 7^2023 mod 10：")
period_7, cycle_7 = find_power_cycle(7, 10)
print(f"  7^n mod 10 的周期 = {period_7}")
print(f"  循环: {cycle_7}")
r = 2023 % period_7
if r == 0:
    r = period_7
answer = cycle_7[r - 1]
print(f"  2023 mod {period_7} = {2023 % period_7}")
print(f"  7^{2023 % period_7} mod 10 = {answer}")

# Python 直接验证
direct = pow(7, 2023, 10)
print(f"\n🔬 Python 直接验证: pow(7, 2023, 10) = {direct}")
print(f"  结果一致? {answer == direct} ✅")

print(f"\n💡 7^2023 的最后一位数字是 {answer}")

In [ ]:
# ========== 步骤 13: 更多幂次末位数字练习 ==========
print("📊 步骤 13: 幂次末位数字综合练习")
print("=" * 55)

problems = [
    (3, 1000, 10, "3^1000 的最后一位数字"),
    (2, 100, 10, "2^100 的最后一位数字"),
    (13, 2023, 100, "13^2023 的最后两位数字"),
    (99, 99, 1000, "99^99 的最后三位数字"),
]

for a, n_pow, m, desc in problems:
    period, cycle = find_power_cycle(a, m)
    r = n_pow % period
    if r == 0:
        r = period
    result = cycle[r - 1]
    direct_check = pow(a, n_pow, m)
    
    print(f"\n  {desc}:")
    print(f"    {a}^n mod {m} 的周期 = {period}")
    print(f"    {n_pow} mod {period} = {n_pow % period}")
    print(f"    答案 = {result}")
    print(f"    Python 验证: {direct_check}  {'✅' if result == direct_check else '❌'}")

print(f"\n💡 掌握了周期性，任何'最后 N 位数字'问题都能快速解决")

In [ ]:
# ========== 步骤 14: 幂次周期可视化 ==========
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

bases = [2, 3, 7, 9]
mod_val = 10

for idx, base in enumerate(bases):
    ax = axes[idx // 2][idx % 2]
    n_show = 20
    values = [pow(base, n, mod_val) for n in range(1, n_show + 1)]
    
    period, cycle = find_power_cycle(base, mod_val)
    
    # 用颜色区分周期
    colors_cycle = [plt.cm.Set1(n % period / max(period - 1, 1)) for n in range(n_show)]
    
    bars = ax.bar(range(1, n_show + 1), values, color=colors_cycle,
                  edgecolor='black', alpha=0.8)
    
    # 标注值
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.15,
                str(v), ha='center', va='bottom', fontweight='bold', fontsize=9)
    
    # 画周期分隔线
    for p in range(period, n_show + 1, period):
        ax.axvline(x=p + 0.5, color='#e74c3c', linestyle='--', alpha=0.5)
    
    ax.set_xlabel('n (exponent)', fontsize=11)
    ax.set_ylabel(f'{base}^n mod {mod_val}', fontsize=11)
    ax.set_title(f'{base}^n mod {mod_val} (period = {period})', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 11)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  每个子图展示不同底数的 a^n mod 10 序列")
print(f"  红色虚线标注周期边界")
print(f"  2: 周期4 (2,4,8,6), 3: 周期4 (3,9,7,1), 7: 周期4 (7,9,3,1), 9: 周期2 (9,1)")

---

## 7. 综合应用：模运算的更多技巧

### 🎯 整除性判断

模运算也是判断整除性的利器：

| 问题 | 转化为 |
|------|--------|
| $n$ 能被 3 整除吗？ | 各位数之和 mod 3 = 0? |
| $n$ 能被 9 整除吗？ | 各位数之和 mod 9 = 0? |
| $n$ 能被 11 整除吗？ | 奇偶位数字交替和 mod 11 = 0? |

In [ ]:
# ========== 步骤 15: 整除性判断与验证 ==========
print("📊 步骤 15: 整除性判断")
print("=" * 55)

def digit_sum(n):
    return sum(int(d) for d in str(abs(n)))

def alternating_digit_sum(n):
    """奇偶位交替和 (从右到左)"""
    digits = [int(d) for d in str(abs(n))]
    digits.reverse()
    return sum(d * (1 if i % 2 == 0 else -1) for i, d in enumerate(digits))

test_numbers = [123456789, 1234567890, 99999, 918273645]

print(f"  {'数字':>15} {'各位和':>8} {'mod 3':>6} {'被3整除':>8} {'mod 9':>6} {'被9整除':>8} {'交替和':>8} {'mod 11':>7} {'被11整除':>9}")
print(f"  {'─'*85}")
for n in test_numbers:
    ds = digit_sum(n)
    ads = alternating_digit_sum(n)
    print(f"  {n:>15} {ds:>8} {ds % 3:>6} {'✅' if n % 3 == 0 else '❌':>8} {ds % 9:>6} {'✅' if n % 9 == 0 else '❌':>8} {ads:>8} {ads % 11:>7} {'✅' if n % 11 == 0 else '❌':>9}")

# 验证
print(f"\n🔬 验证：")
for n in test_numbers:
    ds = digit_sum(n)
    ads = alternating_digit_sum(n)
    rule3_match = (ds % 3 == 0) == (n % 3 == 0)
    rule9_match = (ds % 9 == 0) == (n % 9 == 0)
    rule11_match = (ads % 11 == 0) == (n % 11 == 0)
    print(f"  {n}: 3的规则{'✅' if rule3_match else '❌'}, 9的规则{'✅' if rule9_match else '❌'}, 11的规则{'✅' if rule11_match else '❌'}")

print(f"\n💡 这些整除规则的本质都是模运算:")
print(f"   10 ≡ 1 (mod 3) → 10^k ≡ 1 (mod 3) → 各位数之和 ≡ 原数 (mod 3)")
print(f"   10 ≡ -1 (mod 11) → 10^k ≡ (-1)^k (mod 11) → 交替和 ≡ 原数 (mod 11)")

In [ ]:
# ========== 步骤 16: 模运算在密码学中的应用 (RSA 简化版) ==========
print("📊 步骤 16: 模运算在密码学中的应用")
print("=" * 55)

# RSA 简化版: 用小素数演示
p, q = 17, 23
n_rsa = p * q
phi_n = (p - 1) * (q - 1)  # 欧拉函数

# 选择公钥 e (与 phi(n) 互素)
e_rsa = 3
# 计算私钥 d: e*d ≡ 1 (mod phi(n))
d_rsa = pow(e_rsa, -1, phi_n)  # Python 3.8+ 支持模逆

print(f"  RSA 密钥生成 (简化版):")
print(f"    素数: p = {p}, q = {q}")
print(f"    n = p × q = {n_rsa}")
print(f"    φ(n) = (p-1)(q-1) = {phi_n}")
print(f"    公钥: e = {e_rsa}")
print(f"    私钥: d = {d_rsa}  (因为 {e_rsa} × {d_rsa} = {e_rsa * d_rsa} ≡ {(e_rsa * d_rsa) % phi_n} mod {phi_n})")

# 加密和解密
message = 42
encrypted = pow(message, e_rsa, n_rsa)
decrypted = pow(encrypted, d_rsa, n_rsa)

print(f"\n  加密/解密演示:")
print(f"    原始消息: M = {message}")
print(f"    加密: C = M^e mod n = {message}^{e_rsa} mod {n_rsa} = {message**e_rsa} mod {n_rsa} = {encrypted}")
print(f"    解密: M' = C^d mod n = {encrypted}^{d_rsa} mod {n_rsa} = {decrypted}")
print(f"    解密成功? {decrypted == message} ✅")

# 批量验证
print(f"\n  批量验证 (所有消息 0 到 {n_rsa - 1}):")
all_correct = all(pow(pow(m, e_rsa, n_rsa), d_rsa, n_rsa) == m for m in range(n_rsa))
print(f"    所有 {n_rsa} 条消息加密后解密正确? {all_correct} ✅")

print(f"\n💡 RSA 的安全性依赖于大数分解的困难性")
print(f"   但其数学基础就是模运算: M^(e×d) ≡ M (mod n)")
print(f"   这来自欧拉定理: a^φ(n) ≡ 1 (mod n)")

---

## 8. 核心概念回顾

### 📌 模运算 (Modular Arithmetic)
- **定义**: $a \bmod n = r$，其中 $a = qn + r, \; 0 \le r < n$
- **核心性质**: 对加法、乘法、幂次封闭
- **含义**: 只关注余数，将无穷整数映射到有限集合 $\{0, 1, \ldots, n-1\}$

### 📌 模不变量 (Mod Invariant)
- **定义**: 在某种操作下，某个量的模值保持不变
- **应用**: 变色龙问题中 $(R - G) \bmod 3$ 是不变量
- **含义**: 如果目标状态的不变量值与初始状态不同，则目标不可达

### 📌 信息编码 (Information Encoding)
- **定义**: 用模运算将 $k$ 种可能性编码为一个 $\{0, \ldots, k-1\}$ 的值
- **应用**: 囚犯帽子问题中，第一个人用 $S \bmod k$ 编码全局信息
- **含义**: 一个牺牲者 + 模编码 = 所有人的最优策略

### 📌 周期性 (Periodicity)
- **定义**: $a^n \bmod m$ 随 $n$ 增长呈周期性
- **公式**: 周期 $T$ 满足 $a^{n+T} \equiv a^n \pmod{m}$
- **应用**: 快速计算大幂次的末位数字

### 📌 约瑟夫递推 (Josephus Recurrence)
- **定义**: $J(n, k) = [J(n-1, k) + k] \bmod n$
- **含义**: 淘汰一人后，编号偏移 $k$，问题规模减 1
- **判断标准**: $O(n)$ 时间复杂度，远优于 $O(nk)$ 的直接模拟

### 🔗 完整流程
```
识别模结构 → 构建不变量/编码 → 分析推导 → Python 验证
     ↓              ↓                ↓            ↓
  取余/周期     mod n 不变      数学证明     Monte Carlo
```

### 📝 工具汇总

| 工具 | 用途 | 典型问题 |
|------|------|----------|
| $a \bmod n$ | 取余数 | 末位数字 |
| 模不变量 | 证明不可能 | 变色龙问题 |
| 模编码 | 信息传递 | 囚犯帽子 |
| 模递推 | 缩小问题规模 | 约瑟夫环 |
| 模幂周期 | 快速计算 | $a^n \bmod m$ |

---

## 9. 常见误区

### ❌ 误区 1: 模运算对减法和除法也总是"直接取模"
**✓ 正确理解**: 加法和乘法可以先取模再运算，但**减法需注意负数**（Python 的 `%` 总是返回非负值，但其他语言不一定），**除法必须用模逆元**而非直接取模。$a/b \bmod n$ 不等于 $(a \bmod n) / (b \bmod n)$。

### ❌ 误区 2: 不变量不变就说明问题无解
**✓ 正确理解**: 不变量匹配是**必要条件而非充分条件**。如果不变量不匹配，则一定无解；但不变量匹配时，问题可能有解也可能无解（可能需要更精细的不变量）。

### ❌ 误区 3: 囚犯帽子问题中每个人都有 $1/k$ 的出错概率
**✓ 正确理解**: 只有**最后一个人**（第一个猜的人）有 $1/k$ 的出错概率。其余 99 人利用模信息 + 可见信息，**确定性地**推断出自己的帽子颜色，出错概率为 0。

### ❌ 误区 4: 约瑟夫问题只能用模拟解
**✓ 正确理解**: 约瑟夫问题有 $O(n)$ 的递推公式。当 $k=2$ 时甚至有封闭公式：$J(n,2) = 2L + 1$，其中 $n = 2^m + L$。模拟的时间复杂度是 $O(nk)$，在 $n$ 和 $k$ 很大时远不如递推高效。

### ❌ 误区 5: $a^n \bmod m$ 的周期等于 $m$
**✓ 正确理解**: 周期可以远小于 $m$。例如 $7^n \bmod 10$ 的周期是 4，不是 10。周期等于使 $a^T \equiv 1 \pmod{m}$ 的最小 $T$（当 $\gcd(a, m) = 1$ 时），由欧拉定理知 $T | \varphi(m)$。